In [2]:
import numpy as np
import numpy.linalg as linal
import scipy.io 
import matplotlib.pyplot as plt

In [3]:
data = scipy.io.loadmat('networkdata.mat')
mean_shift = lambda x : x - x.mean(axis=0)

In [4]:
Zt = data['Ftrues']

In [5]:
U, S, V = linal.svd(mean_shift(Zt)/np.sqrt(1000))

In [6]:
S**2

array([8.15203306e+02, 1.92650512e+02, 1.14369743e+02, 8.26508931e+01,
       7.94106319e+01, 5.79822052e+01, 4.09868241e+01, 3.24699156e+01,
       2.43668447e+01, 2.11866554e+01, 2.01050395e+01, 1.91526184e+01,
       1.85695430e+01, 1.82650909e+01, 1.73247080e+01, 1.16070995e+01,
       6.45671872e+00, 5.23719632e-28, 1.35023963e-28, 1.09101341e-28,
       3.89209086e-29, 2.23062511e-29, 1.11473892e-29, 1.05796685e-29,
       6.71614356e-30, 6.55631009e-30])

* 9 Constraints exist from the PCA of true data matrix 
* Because for the uncorrupted data, the constraints lie in the nullspace of the data matrix 

In [7]:
Atrue = V[-9:,:]
# True Constraint matrix
Atrue

array([[ 0.07928232, -0.00335369,  0.01191248,  0.47833438,  0.19802043,
        -0.49487427, -0.20137412,  0.06073774, -0.21594445, -0.23248433,
         0.26238993,  0.20137412,  0.01854458,  0.01256563,  0.03045707,
        -0.06073774,  0.02189827,  0.46376406, -0.00335369, -0.00304957,
         0.0344639 ,  0.03111021,  0.00065314, -0.01283717,  0.00370271,
         0.05738405],
       [-0.00626897, -0.03963596,  0.06803833,  0.19890559,  0.13992886,
         0.27498016, -0.17956482,  0.01097277, -0.6235854 , -0.14969965,
        -0.42467981,  0.17956482, -0.01724174, -0.01262343,  0.05079659,
        -0.01097277,  0.02239422, -0.24511499, -0.03963596,  0.11810249,
         0.00977079, -0.02986517, -0.08066176,  0.2751215 , -0.19876425,
        -0.02866319],
       [-0.02803388, -0.07697585, -0.07327448, -0.4367357 ,  0.32232334,
         0.03455225, -0.39929919,  0.05968107,  0.17346878, -0.22871467,
        -0.26326692,  0.39929919, -0.08771496, -0.08286956, -0.16098944,
       

---
B

In [32]:
from itertools import combinations
comb = list(combinations([i for i in range(26)],9))

val = 1000
index = 0 

for i in range(len(comb)) :
    Ad = Atrue[:,comb[i]]
    # print(f"Condition Number of Ad : {linal.cond(Ad)} \t|\t combination index : {i}")
    test = linal.cond(Ad)

    if test < val : 
        val = test
        index = i

In [40]:
comb[index]
# The smallest condition number occurs for this combination as the dependent variables 
# The condition number is important because it would tell if the matrix Ad is invertible or not 
# If the matrix Ad is not invertible then it will have a very high condition number 
# And it will not be possible to calculate the dependent variables from the regression matrix method

(1, 7, 8, 10, 11, 12, 14, 19, 21)

In [42]:
Ad = Atrue[:,comb[index]]
Ai = Atrue[:,[0,2,3,4,5,6,9,13,15,16,17,18,20,22,23,24,25]]

In [47]:
regression_matrix = -linal.inv(Ad)@Ai
# this regression matrix is to be used with the mean shifted values 

In [48]:
np.shape(regression_matrix)

(9, 17)

---
C

In [50]:
from scipy.stats.distributions import chi2

In [49]:
Z = data['Fmeass']
Zm = mean_shift(Z)

In [53]:
p = 26
N = 1000
d = p-1
Y = Zm      # @np.linalg.inv(L.T)

evals = np.sort(np.linalg.eig( Y.T@Y / N) [0]).real [::-1]
while d > 1 : 
    n_dash = N - (2*p+11)/6
    l_dash = evals[ p-d : ].sum()/d 
    tau    = n_dash * ( d * np.log(l_dash) - np.log(evals[ p-d : ]).sum())
    fdom   = 0.5*(d+2)*(d-1)
    chi    = chi2.ppf(0.95, df=fdom)

    if d <= 12 and d >= 5 : 
        print(f"{d} \t||\t  TEST STATISTIC : {np.round(tau,3)} \t | \t CHI2 : {np.round(chi,3)}")

    # if (tau <= chi) : 
    #     print()
    #     print("Number of Constraints :", d)
        # break 
    # else : 
    d = d - 1 

12 	||	  TEST STATISTIC : 30755.585 	 | 	 CHI2 : 98.484
11 	||	  TEST STATISTIC : 26287.596 	 | 	 CHI2 : 84.821
10 	||	  TEST STATISTIC : 19543.92 	 | 	 CHI2 : 72.153
9 	||	  TEST STATISTIC : 55.21 	 | 	 CHI2 : 60.481
8 	||	  TEST STATISTIC : 34.459 	 | 	 CHI2 : 49.802
7 	||	  TEST STATISTIC : 25.241 	 | 	 CHI2 : 40.113
6 	||	  TEST STATISTIC : 15.871 	 | 	 CHI2 : 31.41
5 	||	  TEST STATISTIC : 9.995 	 | 	 CHI2 : 23.685


* Hypothesis testing is capable of estimating the correct number of constraints 
* Estimated number of constraints = 9 

In [56]:
evals[-9:].mean()

0.06202667219753946

In [57]:
U1, S1, V1 = linal.svd(Z/np.sqrt(N))

* The value of error variance is the average of all the eigenvalues that form the constraint matrix 
* error variance = 0.06203
---

In [64]:
Acons = V1[-9:]

In [65]:
Ad = Acons[:,comb[index]]
Ai = Acons[:,[0,2,3,4,5,6,9,13,15,16,17,18,20,22,23,24,25]]
regression_matrix_meas = -linal.inv(Ad)@Ai

In [66]:
np.abs(regression_matrix - regression_matrix_meas).max()

0.019832798290010504

* The maximum value of absolute error is 0.0198327
---